# ERK-KTR whole-FOV stimulation on the Ti2-E: drug test with pre-exposure (df_acquire API)

A multi-phase ERK-KTR experiment with whole-FOV blue light, built with
the **legacy `df_acquire` API** (`generate_df_acquire()` +
`apply_stim_treatments_to_df_acquire()`, then `df_to_events()`).

**Design:**
- 3 FOV groups (about 21 FOVs each), each with its own set of stimulation treatments
- Each group is split at timestep 100 into a pre-drug and a post-drug half
- The drug is pipetted by hand between the halves; the notebook pauses at cell boundaries
- Treatments differ in drug concentration, pre-stimulation and exposure (including a ramp)

**Workflow:**
1. Connect to the Ti2-E and define the channels
2. Define the stimulation treatments
3. Set up the pipeline
4. Pick FOV positions in napari
5. Build one `df_acquire` per FOV group and split it into pre- and post-drug halves
6. Run the halves in sequence, pipetting between them
7. Merge the tracks into `exp_data.parquet`

**Channels:**

| Role | Jungfrau preset | Ti2-E preset |
|------|-----------------|--------------|
| Nuclear marker (H2B-miRFP), segmented | `miRFP` | `iRFP` |
| ERK-KTR reporter (mScarlet3) | `mScarlet3` | `RFP` |
| Stimulation, blue light, whole FOV | `CyanStim` | `GFP` |
| Optocheck (mCitrine) | `mCitrine` | `YFP_Custom` |

The Ti2-E has no DMD, so stimulation lights the whole field of view with the
blue excitation of the ET-GFP cube. The optocheck channel is excited by the
same cube, so it also stimulates; that is why it runs on the last frame only.
All presets share the X-Cite XT600 LED; the `power` of a `PowerChannel` sets
its intensity in percent. Exposures and powers are the Jungfrau values:
tune them for the XT600 before a real run.

In [ ]:
import os
import time

import numpy as np
import pandas as pd

from faro.core.data_structures import (
    Channel,  # imaging channel (config, exposure, group)
    PowerChannel,  # channel with light-source power (adds power 0-100)
    StimTreatment,  # stimulation schedule of one treatment
    SegmentationMethod,
)
import faro.core.utils as utils

### Microscope

`Ti2E()` loads `NikonTi2e.cfg`, applies `System/Startup` and selects the
`Channel` group, so channels need no `group`. Close NIS-Elements first.

In [ ]:
from faro_aoki.ti2e import Ti2E

mic = Ti2E()
utils.print_configs(mic.mmc)

### Experimental settings

- `N_FRAMES` / `TIME_BETWEEN_TIMESTEPS`: frames per FOV group and interval
- `base_path` / `experiment_name`: output folder; `WRITER`: `"omezarr"` or `"tiff"`
- `channels`: imaging channels on every frame (channel 0 is segmented)
- `channel_optocheck`: reference channel on the last frame
- `condition`: condition labels assigned to the FOVs
- `n_fovs_per_well`: FOVs per well on a plate, `None` for free FOVs

In [ ]:
N_FRAMES = 180  # per FOV group; split at timestep 100 into pre/post-drug
TIME_BETWEEN_TIMESTEPS = 60  # s
TIME_PER_FOV = 2  # s: exposures + stage move + overhead

ADD_STIM_EXPOSURE_GROUP = False
REGULAR_SPACING_BETWEEN_STIMULATIONS = False

## Storage
base_path = os.path.join(os.path.expanduser("~"), "faro_data")  # TODO: data folder
experiment_name = "2026-11-16_SUSTAINED_DIFF.DENS_RMC6236"
path = os.path.join(base_path, experiment_name)
# TIFF: on this faro version OmeZarrWriter cannot add FOVs in
# continue_experiment, so it would drop FOV groups 2 and 3.
WRITER = "tiff"

## Imaging channels, on every frame (channel 0 is segmented)
channels = [
    PowerChannel(
        config="iRFP", exposure=125, group=mic.CHANNEL_GROUP, power=95
    ),  # H2B-miRFP
    PowerChannel(
        config="RFP", exposure=125, group=mic.CHANNEL_GROUP, power=95
    ),  # ERK-KTR-mScarlet3
]

## Optocheck on the last frame
channel_optocheck = PowerChannel(
    config="YFP_Custom", exposure=600, group=mic.CHANNEL_GROUP, power=99
)
optocheck_timepoints = (N_FRAMES - 1,)

condition = ["EGFR"]
n_fovs_per_well = 3

### Stimulation treatments

Each `StimTreatment` sets when and how a group of FOVs is stimulated:
- `treatment_name`: label, parsed later into analysis columns
- `stim_timestep`: frames with stimulation
- `stim_exposure_list`: exposure(s) in ms, one value or one per frame (ramps)
- `stim_power`: LED intensity (0-100)
- `stim_channel_*`: preset, group, and the device/property of the power
- `auto_repeat_stim_exposure`: repeat a single exposure over all `stim_timestep`

On the Ti2-E the stimulation is the `GFP` preset, and its power is the
XT600 LED intensity (`mic.POWER_PROPERTIES`).

**Timestep library:** `with_prestim` stimulates frames 10-99 (pre-drug)
and 101-159 (post-drug); `without_prestim` only 101-159.

**Helpers:**
- `build_stim_treatments()` builds `StimTreatment` objects from compact specs
- `parse_treatment_name()` splits the treatment name into drug concentration, exposure and pre-stimulation columns
- `split_df_acquire()` splits a df_acquire at a timestep and restarts the time of the second half at 0

In [ ]:
with_prestim_timesteps = tuple(range(10, 100)) + tuple(range(101, 160))
without_prestim_timesteps = tuple(range(101, 160))

# (prestim sign, interval label) -> frames
timestep_lookup = {
    ("+", "1min"): with_prestim_timesteps,
    ("-", "1min"): without_prestim_timesteps,
}

LED_DEVICE, LED_INTENSITY = mic.POWER_PROPERTIES["GFP"]


def build_stim_treatments(specs, timestep_lookup, *, stim_power=10):
    """Build StimTreatment objects from (drug_nM, prestim, timing_key, exposure_ms) specs.

    exposure_ms is an int, or "ru" for the ramp-up protocol: exposure
    rising pre-drug, then constant post-drug.
    """
    treatments = []
    for drug_nM, prestim, timing_key, exposure_ms in specs:
        sign = "+" if prestim else "-"
        key = (sign, timing_key)
        if key not in timestep_lookup:
            raise ValueError(f"Unknown timing config: {key}")
        if exposure_ms != "ru" and not isinstance(exposure_ms, int):
            raise ValueError(f"Exposure must be 'ru' or int, got {exposure_ms}")

        if exposure_ms == "ru":
            if prestim:
                exposure_ms = tuple(range(0, 900, 10)) + (250,) * 60
            else:
                exposure_ms = (250,) * 60
            treatment_name = f"{drug_nM}nM -1ms {sign} stim{timing_key}"
        else:
            treatment_name = f"{drug_nM}nM {exposure_ms}ms {sign} stim{timing_key}"

        print(treatment_name)
        treatments.append(
            StimTreatment(
                treatment_name=treatment_name,
                stim_timestep=timestep_lookup[key],
                stim_exposure_list=exposure_ms,
                stim_power=stim_power,
                stim_channel_name="GFP",
                stim_channel_group=mic.CHANNEL_GROUP,
                stim_channel_device_name=LED_DEVICE,
                stim_channel_power_property_name=LED_INTENSITY,
                auto_repeat_stim_exposure=True,
            )
        )
    return treatments


def parse_treatment_name(df_acquire):
    """Split "{drug}nM {exposure}ms {+/-} stim{interval}" into analysis columns."""
    parts = df_acquire["treatment_name"].str.split(" ", expand=True)
    df_acquire["drug_concentration"] = parts[0].str.replace("nM", "").astype(int)
    if "ru" in parts[1].iloc[0]:
        df_acquire["pattern_exposure"] = np.nan
    else:
        df_acquire["pattern_exposure"] = parts[1].str.replace("ms", "").astype(float)
    df_acquire["prestimulation"] = parts[2] == "+"
    df_acquire["stim_every_x_min"] = (
        parts[3].str.replace("stim", "").str.replace("min", "").astype(int)
    )
    return df_acquire


def split_df_acquire(df, split_timestep):
    """Split at split_timestep; time and timestep of the second half restart at 0.

    continue_experiment() appends the second half after the frames
    already acquired, so its timesteps must start at 0 again; the
    file names keep the original timesteps.
    """
    t_offset = df.loc[df["timestep"] == split_timestep, "time"].iloc[0]
    df_1 = df[df["timestep"] < split_timestep]
    df_2 = df[df["timestep"] >= split_timestep].copy()
    df_2["time"] = df_2["time"] - t_offset
    df_2["timestep"] = df_2["timestep"] - split_timestep
    return df_1, df_2

In [ ]:
# Treatment specs per FOV group: (drug_nM, prestim, timing_key, exposure_ms)

# Group 1: FOVs 0-20
stim_phase_1 = build_stim_treatments(
    [
        (300, False, "1min", 250),
        (300, True, "1min", 250),
        (500, False, "1min", 250),
        (500, True, "1min", 250),
        (0, False, "1min", 250),  # control
        (0, True, "1min", 250),
        (300, False, "1min", 250),
    ],
    timestep_lookup,
)

# Group 2: FOVs 21-41
stim_phase_2 = build_stim_treatments(
    [
        (300, True, "1min", 250),
        (500, False, "1min", 250),
        (500, True, "1min", 250),
        (0, False, "1min", 250),
        (0, True, "1min", 250),
        (300, False, "1min", 250),
        (500, False, "1min", 250),
    ],
    timestep_lookup,
)

# Group 3: FOVs 42 and up
stim_phase_3 = build_stim_treatments(
    [
        (0, False, "1min", 250),
        (300, True, "1min", 250),
        (500, True, "1min", 250),
        (300, False, "1min", 250),
        (500, False, "1min", 250),
        (300, True, "1min", 250),
        (500, True, "1min", 250),
    ],
    timestep_lookup,
)

### Pipeline

In [ ]:
from faro.core.controller import Controller
from faro.core.conversion import df_to_events
from faro.core.pipeline import ImageProcessingPipeline
from faro.core.writers import OmeZarrWriter, TiffWriter
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.ref import RefFE
from faro.segmentation.cellpose_v4 import CellposeV4
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy

# Cellpose needs `uv sync --extra cellpose`. Without it, OtsuSegmentator works
# for a bright nuclear marker:
# from faro.segmentation.base import OtsuSegmentator;  CellposeV4() -> OtsuSegmentator()

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=CellposeV4(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimWholeFOV(),
    feature_extractor_ref=RefFE(used_mask="labels"),
)

if WRITER == "omezarr":
    writer = OmeZarrWriter(storage_path=path)
else:
    writer = TiffWriter(storage_path=path)

### GUI: napari and Micro-Manager

Opens napari with the napari-micromanager widget. Use it for live view and
to pick the FOV positions in the MDA widget; the next steps read them from
there.

In [ ]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer, mmcore=mic.mmc)
viewer.window.add_dock_widget(mm_wdg)

#### Sort the FOV positions by well column

Expects position names like `B3_0000` from the well-plate tab of the
MDA widget.

In [ ]:
mda_widget = viewer.window.dock_widgets["MDA"]
fov_pos = mda_widget.value().stage_positions


def sort_key(fov):
    """Sort by well number, then letter, then index within the well."""
    well, idx = fov.name.split("_", 1)
    return (int(well[1:]), well[0], idx)


fovs_sorted = sorted(fov_pos, key=sort_key)
mda_widget.setValue(
    mda_widget.value().model_copy(update={"stage_positions": fovs_sorted})
)

#### Plot the FOV positions

In [ ]:
import matplotlib.pyplot as plt

fovs = utils.generate_fov_positions(mic, viewer=viewer)

plt.figure(figsize=(8, 6))
plt.scatter([f.x for f in fovs], [f.y for f in fovs])
for i, f in enumerate(fovs):
    plt.annotate(
        str(i), (f.x, f.y), textcoords="offset points", xytext=(5, 5), fontsize=7
    )
plt.xlabel("X (um)")
plt.ylabel("Y (um)")
plt.title("FOV positions")
plt.axis("equal")
plt.tight_layout()
plt.show()
print(f"{len(fovs)} FOVs selected")

### Build the df_acquire tables

Each FOV group gets its own df_acquire with its own treatments.
`first_fov_index` keeps the FOV indices of the groups apart, so their
files and tracks never collide. Each table is split at timestep 100
into a pre-drug and a post-drug half.

In [ ]:
# FOV group 1: FOVs 0-20
group_fovs = fovs[0:21]
print(f"FOV group 1: {len(group_fovs)} FOVs")

df_acquire_1 = utils.generate_df_acquire(
    group_fovs,
    first_fov_index=0,  # keeps the FOV indices of the groups apart
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
    optocheck_timepoints=optocheck_timepoints,
    channel_optocheck=channel_optocheck,
)
df_acquire_1 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_1,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_1 = parse_treatment_name(df_acquire_1)

# Split at timestep 100 into pre-drug and post-drug
df_acquire_1_1, df_acquire_1_2 = split_df_acquire(df_acquire_1, 100)

In [ ]:
# FOV group 2: FOVs 21-41
group_fovs = fovs[21:42]
print(f"FOV group 2: {len(group_fovs)} FOVs")

df_acquire_2 = utils.generate_df_acquire(
    group_fovs,
    first_fov_index=21,  # keeps the FOV indices of the groups apart
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
    optocheck_timepoints=optocheck_timepoints,
    channel_optocheck=channel_optocheck,
)
df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2 = parse_treatment_name(df_acquire_2)

# Split at timestep 100 into pre-drug and post-drug
df_acquire_2_1, df_acquire_2_2 = split_df_acquire(df_acquire_2, 100)

In [ ]:
# FOV group 3: FOVs 42 and up
group_fovs = fovs[42:]
print(f"FOV group 3: {len(group_fovs)} FOVs")

df_acquire_3 = utils.generate_df_acquire(
    group_fovs,
    first_fov_index=42,  # keeps the FOV indices of the groups apart
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
    optocheck_timepoints=optocheck_timepoints,
    channel_optocheck=channel_optocheck,
)
df_acquire_3 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_3,
    stim_phase_3,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_3 = parse_treatment_name(df_acquire_3)

# Split at timestep 100 into pre-drug and post-drug
df_acquire_3_1, df_acquire_3_2 = split_df_acquire(df_acquire_3, 100)

### Run the experiment

Six halves in sequence, with drug pipetting in between:
1. `df_acquire_1_1`: group 1 pre-drug, timed so it ends at `start_at`
2. *pipette the drug for group 1*
3. `df_acquire_1_2` + `df_acquire_2_1`: group 1 post-drug, group 2 pre-drug
4. *pipette the drug for group 2*
5. `df_acquire_2_2` + `df_acquire_3_1`: group 2 post-drug, group 3 pre-drug
6. *pipette the drug for group 3*
7. `df_acquire_3_2`: group 3 post-drug, then finish

The first half uses `run_experiment`; all later ones use
`continue_experiment`, which keeps tracking state, timestep counters
and file names.

**Known limits of this faro version:**
- FOV groups 2 and 3 start after other frames were acquired. For their first frame the pipeline waits for an earlier frame of the same FOV that never comes, logs `FrameDispenser: timeout waiting for frame N` after 20 s, and then goes on. The data are complete, but expect these errors and some lag at the start of each new group.
- `OmeZarrWriter` sizes its position axis at the first run and cannot add the FOVs of groups 2 and 3 later (fixed on the faro branch `fix/omezarr-grow-positions`, not merged yet). Keep `WRITER = "tiff"` until then.

In [ ]:
from datetime import datetime

## Optional: time phase 1 so it ends at the first drug addition
start_at = None  # e.g. "2026-11-17 08:45:00"

if start_at is not None:
    wait_seconds = (
        datetime.strptime(start_at, "%Y-%m-%d %H:%M:%S") - datetime.now()
    ).total_seconds() - df_acquire_1_1.time.max()
    if wait_seconds > 0:
        print(f"Waiting {wait_seconds / 60:.1f} min, 1st pipetting at {start_at}")
        for _ in range(int(wait_seconds)):
            time.sleep(1)

print("Starting experiment")
ctrl = Controller(mic, pipeline, writer=writer)
ctrl.run_experiment(df_to_events(df_acquire_1_1), stim_mode="current").wait()

# Pause point: pipette the drug for group 1, then run the next cell.

In [ ]:
from faro.widgets import ExperimentStatusWidget

viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)

In [ ]:
## Group 1 post-drug + group 2 pre-drug
ctrl.continue_experiment(df_to_events(df_acquire_1_2), stim_mode="current").wait()
ctrl.continue_experiment(df_to_events(df_acquire_2_1), stim_mode="current").wait()

# Pause point: pipette the drug for group 2, then run the next cell.

In [ ]:
## Group 2 post-drug + group 3 pre-drug
ctrl.continue_experiment(df_to_events(df_acquire_2_2), stim_mode="current").wait()
ctrl.continue_experiment(df_to_events(df_acquire_3_1), stim_mode="current").wait()

# Pause point: pipette the drug for group 3, then run the next cell.

In [ ]:
## Group 3 post-drug, then finish
ctrl.continue_experiment(df_to_events(df_acquire_3_2), stim_mode="current").wait()

ctrl.finish_experiment()  # drain the pipeline and close the writer
mic.post_experiment()
utils.generate_exp_data_from_tracks(path)
print("Experiment complete!")